# <font color="#418FDE" size="6.5" uppercase>**Robust Sparse**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Apply sparse linear estimators for feature selection and compact prediction. 
- Use Bayesian linear models to estimate regularized coefficients and uncertainty-related behavior. 
- Fit robust regressors that reduce sensitivity to outliers. 


## **1. Sparse Linear Methods**

### **1.1. Least Angle Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_01.jpg?v=1783993515" width="250">



>* Builds sparse models by adding strongest predictors
>* Creates a path of growing model complexity

>* Efficiently explores many sparse model choices
>* Balances accuracy, interpretability, and feature costs

>* Balances cautious and aggressive model building
>* Shows feature importance through sparse paths



In [ ]:
#@title Python Code - Least Angle Regression

# Demonstrate Least Angle Regression for sparse prediction.
# Track how features enter the model path.
# Compare selected coefficients with true useful features.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import Lars
from sklearn.preprocessing import StandardScaler

# Create data with many features but few true signals.
features, target, true_coefficients = make_regression(
    n_samples=120,
    n_features=12,
    n_informative=3,
    coef=True,
    noise=8.0,
    random_state=42,
)

# Validate the generated teaching dataset shape.
if features.shape != (120, 12):
    raise ValueError("Expected 120 rows and 12 feature columns.")

# Standardizing makes coefficient paths easier to compare.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Fit one LARS model and keep its coefficient path.
model = Lars(n_nonzero_coefs=6, fit_path=True)
model.fit(scaled_features, target)

# Identify the truly useful generated features.
true_useful = np.flatnonzero(true_coefficients != 0)
selected = np.flatnonzero(model.coef_ != 0)

print("scikit-learn version:", sklearn.__version__)
print("True useful feature numbers:", true_useful.tolist())
print("Features selected by LARS:", selected.tolist())
print("Nonzero coefficients in final compact model:", len(selected))

# Plot how coefficients grow as LARS adds features.
fig, ax = plt.subplots(figsize=(8, 5))
steps = np.arange(model.coef_path_.shape[1])

for feature_number in range(model.coef_path_.shape[0]):
    path = model.coef_path_[feature_number]
    label = "feature " + str(feature_number)
    ax.plot(steps, path, marker="o", linewidth=1.5, label=label)

ax.set_title("Least Angle Regression coefficient path")
ax.set_xlabel("LARS path step")
ax.set_ylabel("Standardized coefficient value")
ax.legend(ncol=2, fontsize=8)
plt.show()



### **1.2. Lasso LARS**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_02.jpg?v=1783993519" width="250">



>* Builds sparse models by selecting key predictors
>* Shows predictor entry as regularization relaxes

>* Builds sparse models along a regularization path
>* Adds or removes predictors to handle overlap

>* Sparse models improve interpretability and deployment
>* Validate choices, especially with correlated predictors



In [ ]:
#@title Python Code - Lasso LARS

# This example demonstrates sparse feature selection.
# Lasso LARS keeps weak coefficients at zero.
# The plot shows the regularization path.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import lars_path
from sklearn.preprocessing import StandardScaler

# Create data with only a few truly useful features.
features, target, true_coefficients = make_regression(
    n_samples=80,
    n_features=10,
    n_informative=3,
    coef=True,
    noise=8.0,
    random_state=42,
)

# Validate the generated data before modeling.
if features.shape != (80, 10):
    raise ValueError("Expected 80 rows and 10 features.")

# Standardization makes coefficient paths easier to compare.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
centered_target = target - np.mean(target)

# Lasso LARS traces coefficients from sparse to richer models.
alphas, active, coefficient_path = lars_path(
    scaled_features,
    centered_target,
    method="lasso",
)

# Count selected features at the final path step.
final_coefficients = coefficient_path[:, -1]
selected_count = int(np.sum(np.abs(final_coefficients) > 0.001))

print("scikit-learn version:", sklearn.__version__)
print("True informative features:", int(np.sum(true_coefficients != 0)))
print("Features selected at path end:", selected_count)

# Plot each feature coefficient as regularization relaxes.
fig, ax = plt.subplots(figsize=(8, 5))
for feature_index in range(coefficient_path.shape[0]):
    ax.plot(alphas, coefficient_path[feature_index], linewidth=1.5)

ax.invert_xaxis()
ax.set_title("Lasso LARS coefficient path")
ax.set_xlabel("Alpha, larger means stronger regularization")
ax.set_ylabel("Standardized coefficient value")
plt.show()



### **1.3. Greedy Sparse Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_03.jpg?v=1783993517" width="250">



>* Stepwise feature choices build compact models
>* Small predictor sets improve interpretability and efficiency

>* Forward selection adds strongest features stepwise
>* Stopping rules prevent overfitting and early bias

>* Compact models aid interpretation and efficiency
>* Validate selections; avoid causal overinterpretation



In [ ]:
#@title Python Code - Greedy Sparse Selection

# This example demonstrates greedy sparse feature selection.
# Forward selection adds useful predictors step by step.
# The final model uses fewer selected features.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Create data where only three features truly matter.
X, y, true_coefficients = make_regression(
    n_samples=220,
    n_features=10,
    n_informative=3,
    coef=True,
    noise=12.0,
    random_state=42,
)

# Check that the generated table has the expected shape.
if X.shape != (220, 10):
    raise ValueError("Expected 220 rows and 10 features.")

# Split data before fitting selection or regression.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
)

# Forward selection greedily chooses three predictive features.
base_model = LinearRegression()
selector = SequentialFeatureSelector(
    base_model,
    n_features_to_select=3,
    direction="forward",
    scoring="r2",
    cv=5,
)

# Fit selection only on training data.
selector.fit(X_train, y_train)
selected_mask = selector.get_support()
selected_features = np.where(selected_mask)[0]

# Fit a compact linear model using only selected features.
compact_model = LinearRegression()
compact_model.fit(X_train[:, selected_mask], y_train)
compact_predictions = compact_model.predict(X_test[:, selected_mask])

# Compare with a model that uses every feature.
full_model = LinearRegression()
full_model.fit(X_train, y_train)
full_predictions = full_model.predict(X_test)

compact_r2 = r2_score(y_test, compact_predictions)
full_r2 = r2_score(y_test, full_predictions)
true_features = np.where(true_coefficients != 0)[0]

print("scikit-learn version:", sklearn.__version__)
print("True informative feature numbers:", true_features.tolist())
print("Greedily selected feature numbers:", selected_features.tolist())
print("Test R2 with 3 selected features:", round(compact_r2, 3))
print("Test R2 with all 10 features:", round(full_r2, 3))

# Plot true coefficients and highlight selected features.
colors = np.where(selected_mask, "tab:orange", "tab:blue")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(10), true_coefficients, color=colors)
ax.set_title("Greedy Forward Selection Finds a Compact Feature Set")
ax.set_xlabel("Feature number")
ax.set_ylabel("True coefficient value")
ax.legend(["Selected features are orange"])
plt.show()



## **2. Bayesian Linear Models**

### **2.1. Bayesian Ridge**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_01.jpg?v=1783993505" width="250">



>* Probabilistic ridge treats coefficients as uncertain
>* Shrinkage improves stability with correlated features

>* Regularization reflects noise and coefficient beliefs
>* Conservative estimates reduce overfitting correlated predictors

>* Shows uncertainty in noisy coefficient estimates
>* Provides stable, interpretable regularized predictions



In [ ]:
#@title Python Code - Bayesian Ridge

# Demonstrate Bayesian Ridge on correlated regression features.
# Regularization shrinks unstable coefficients toward safer values.
# Printed results compare accuracy and coefficient behavior.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small regression dataset with correlated predictors.
rng = np.random.default_rng(42)
base_feature = rng.normal(size=180)
related_feature = base_feature + rng.normal(scale=0.08, size=180)

# Add extra noisy predictors to make regularization useful.
noise_features = rng.normal(size=(180, 4))
X = np.column_stack((base_feature, related_feature, noise_features))
true_coefficients = np.array([3.0, 3.0, 0.0, 0.0, 0.0, 0.0])

y = X @ true_coefficients + rng.normal(scale=1.2, size=180)

# Validate the basic shape before modeling.
if X.shape != (180, 6) or y.shape != (180,):
    raise ValueError("Expected 180 rows, 6 features, and 180 targets.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Fit scaling only on training data to avoid leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Bayesian Ridge estimates regularized coefficients from the data.
model = BayesianRidge()
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)
rmse = mean_squared_error(y_test, predictions) ** 0.5

# Smaller coefficient spread shows conservative regularized behavior.
coefficient_spread = np.max(np.abs(model.coef_)) - np.min(np.abs(model.coef_))
rounded_coefficients = np.round(model.coef_, 2)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Estimated noise precision alpha: {model.alpha_:.2f}")
print(f"Estimated weight precision lambda: {model.lambda_:.2f}")
print(f"Coefficient spread: {coefficient_spread:.2f}")
print(f"Coefficients: {rounded_coefficients}")



### **2.2. ARD Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_02.jpg?v=1783993503" width="250">



>* Bayesian linear model with coefficient-specific shrinkage
>* Selects useful predictors, shrinks noisy ones

>* Feature-specific shrinkage reflects data support
>* Highlights useful predictors, not causal truth

>* Shows coefficient importance and uncertainty
>* Encourages sparse models, but needs caution



In [ ]:
#@title Python Code - ARD Regression

# Demonstrate ARD regression on sparse synthetic data.
# Show coefficient shrinkage and uncertainty estimates.
# Identify useful predictors in a compact model.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import ARDRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Create data where only three predictors truly matter.
X, y, true_coefficients = make_regression(
    n_samples=180,
    n_features=12,
    n_informative=3,
    noise=12.0,
    coef=True,
    random_state=42,
)

# Check the generated data has the expected shape.
if X.shape != (180, 12):
    raise ValueError("Expected 180 rows and 12 features.")

# Split data before scaling to avoid leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
)

# Fit ARD after standardizing feature scales.
model = make_pipeline(
    StandardScaler(),
    ARDRegression(max_iter=300),
)
model.fit(X_train, y_train)

# Extract the fitted Bayesian linear model.
ard_model = model.named_steps["ardregression"]
learned_coefficients = ard_model.coef_
coefficient_uncertainty = np.sqrt(np.diag(ard_model.sigma_))

# Count coefficients that ARD kept meaningfully away from zero.
kept_count = int(np.sum(np.abs(learned_coefficients) > 1.0))
test_r2 = r2_score(y_test, model.predict(X_test))

print("scikit-learn version:", sklearn.__version__)
print("Test R^2:", round(test_r2, 3))
print("Coefficients kept above 1.0:", kept_count, "of", X.shape[1])

# Plot fitted coefficients with uncertainty bars.
feature_numbers = np.arange(1, X.shape[1] + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(
    feature_numbers,
    learned_coefficients,
    yerr=coefficient_uncertainty,
    fmt="o",
    capsize=4,
    label="ARD estimate ± uncertainty",
)

ax.axhline(0, color="black", linewidth=1)
ax.set_title("ARD Regression Shrinks Irrelevant Coefficients")
ax.set_xlabel("Feature number")
ax.set_ylabel("Standardized coefficient")
ax.legend()
plt.show()



### **2.3. Uncertainty Intuition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_03.jpg?v=1783993507" width="250">



>* Coefficients have ranges, not fixed answers
>* Bayesian models reveal uncertainty in estimates

>* Weak evidence shrinks coefficients conservatively
>* Uncertainty helps avoid overfitting noisy patterns

>* Prediction uncertainty grows when extrapolating
>* Confidence levels guide safer decisions



## **3. Robust Regression**

### **3.1. Huber Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_01.jpg?v=1783993513" width="250">



>* Robust linear regression for outlier-prone data
>* Limits extreme points’ influence on trends

>* Small errors count normally; large errors downweighted
>* Threshold balances robustness against clean-data efficiency

>* Handles heavy-tailed noise without deleting outliers
>* Keeps linear models stable and interpretable



In [ ]:
#@title Python Code - Huber Regression

# This example compares ordinary and Huber regression.
# Outliers are added to a simple linear dataset.
# The robust line should follow typical points better.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Create a deterministic one-feature regression dataset.
rng = np.random.default_rng(42)
x_values = np.linspace(0, 10, 80)
noise = rng.normal(0, 1.0, size=x_values.shape)
y_values = 3.0 * x_values + 5.0 + noise

# Add a few large outliers to the target values.
outlier_indices = np.array([8, 18, 55, 70])
y_with_outliers = y_values.copy()
y_with_outliers[outlier_indices] += np.array([28, -24, 30, -26])

# Validate the shapes before fitting the models.
X = x_values.reshape(-1, 1)
if X.shape[0] != y_with_outliers.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# Fit ordinary least squares and robust Huber regression.
ols_model = LinearRegression()
huber_model = HuberRegressor(epsilon=1.35, alpha=0.0, max_iter=200)
ols_model.fit(X, y_with_outliers)
huber_model.fit(X, y_with_outliers)

# Compare errors against the clean trend for teaching clarity.
ols_predictions = ols_model.predict(X)
huber_predictions = huber_model.predict(X)
ols_mae = mean_absolute_error(y_values, ols_predictions)
huber_mae = mean_absolute_error(y_values, huber_predictions)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"OLS MAE versus clean trend: {ols_mae:.2f}")
print(f"Huber MAE versus clean trend: {huber_mae:.2f}")
print(f"Huber marked outliers: {int(huber_model.outliers_.sum())}")

# Plot the data and both fitted lines.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_values, y_with_outliers, label="data with outliers", alpha=0.75)
ax.plot(x_values, ols_predictions, label="ordinary least squares", linewidth=2)
ax.plot(x_values, huber_predictions, label="Huber regression", linewidth=2)

ax.set_title("Huber regression reduces outlier influence")
ax.set_xlabel("Feature value")
ax.set_ylabel("Target value")
ax.legend()
plt.show()



### **3.2. RANSAC Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_02.jpg?v=1783993511" width="250">



>* Fits models from random data subsets
>* Finds inliers despite severe outliers

>* Find models from clean random inlier subsets
>* Keep strongest consensus, separating outliers directly

>* Tune inlier tolerance and trial count carefully
>* Best when outliers are distinct and limited



In [ ]:
#@title Python Code - RANSAC Regression

# Demonstrate RANSAC regression with severe outliers.
# Compare robust fitting against ordinary least squares.
# Show which points RANSAC treats as inliers.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import RANSACRegressor
from sklearn.metrics import mean_absolute_error

# Create a simple distance and delivery time relationship.
rng = np.random.default_rng(42)
distance_km = np.linspace(1, 20, 60)
normal_time_min = 8 + 3 * distance_km + rng.normal(0, 2, 60)

# Add a few corrupted delivery records as severe outliers.
outlier_positions = np.array([8, 18, 35, 48, 55])
observed_time_min = normal_time_min.copy()
observed_time_min[outlier_positions] += np.array([35, -28, 45, -32, 38])

# Validate the generated data before fitting models.
X = distance_km.reshape(-1, 1)
y = observed_time_min
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# Fit ordinary least squares to every observation.
least_squares = LinearRegression()
least_squares.fit(X, y)
least_squares_predictions = least_squares.predict(X)

# Fit RANSAC to find the dominant inlier pattern.
ransac = RANSACRegressor(
    estimator=LinearRegression(),
    residual_threshold=6.0,
    random_state=42,
)
ransac.fit(X, y)
ransac_predictions = ransac.predict(X)

# Compare errors against the known clean relationship.
ols_error = mean_absolute_error(normal_time_min, least_squares_predictions)
ransac_error = mean_absolute_error(normal_time_min, ransac_predictions)
inlier_count = int(np.sum(ransac.inlier_mask_))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"RANSAC inliers found: {inlier_count} of {len(y)} records")
print(f"MAE versus clean trend, OLS: {ols_error:.2f} minutes")
print(f"MAE versus clean trend, RANSAC: {ransac_error:.2f} minutes")

# Plot the data and both fitted regression lines.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X[ransac.inlier_mask_], y[ransac.inlier_mask_], label="RANSAC inliers")
ax.scatter(X[~ransac.inlier_mask_], y[~ransac.inlier_mask_], label="Outliers")
ax.plot(distance_km, least_squares_predictions, label="Ordinary least squares")
ax.plot(distance_km, ransac_predictions, label="RANSAC regression")

ax.set_title("RANSAC reduces the influence of severe outliers")
ax.set_xlabel("Delivery distance (km)")
ax.set_ylabel("Delivery time (minutes)")
ax.legend()
plt.show()



### **3.3. Quantile Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_03.jpg?v=1783993509" width="250">



>* Predict medians or percentiles, not averages
>* Reduce outlier influence on fitted relationships

>* Quantiles show varying predictor effects
>* Models reveal changing spread across outcomes

>* Limits outlier influence without deleting data
>* Models specific quantiles for practical risk insights



In [ ]:
#@title Python Code - Quantile Regression

# Demonstrate quantile regression on outlier affected data.
# Compare median predictions with mean based regression.
# Show robustness through fitted lines and errors.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import QuantileRegressor
from sklearn.metrics import mean_absolute_error

# Create a small deterministic regression dataset.
rng = np.random.default_rng(42)
x_values = np.linspace(0, 10, 80)
noise = rng.normal(0, 1.0, size=x_values.shape)
y_values = 3.0 + 1.8 * x_values + noise

# Add a few large response outliers.
outlier_positions = np.array([8, 18, 55, 70])
y_values[outlier_positions] = y_values[outlier_positions] + 18
X = x_values.reshape(-1, 1)

# Validate the simple shape expected by both models.
if X.shape[0] != y_values.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# Fit one mean model and one median quantile model.
mean_model = LinearRegression()
median_model = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs")
mean_model.fit(X, y_values)
median_model.fit(X, y_values)

# Predict on a smooth grid for easy visual comparison.
x_grid = np.linspace(0, 10, 200).reshape(-1, 1)
mean_predictions = mean_model.predict(x_grid)
median_predictions = median_model.predict(x_grid)

# Compare absolute error on the original observations.
mean_mae = mean_absolute_error(y_values, mean_model.predict(X))
median_mae = mean_absolute_error(y_values, median_model.predict(X))
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Mean regression MAE: {mean_mae:.2f}")
print(f"Median quantile regression MAE: {median_mae:.2f}")

# Plot the data and both fitted relationships.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_values, y_values, color="gray", alpha=0.75, label="data with outliers")
ax.plot(x_grid.ravel(), mean_predictions, color="red", label="mean regression")
ax.plot(x_grid.ravel(), median_predictions, color="blue", label="median quantile regression")

# Label the plot for beginner interpretation.
ax.set_title("Quantile Regression Is Less Pulled by High Outliers")
ax.set_xlabel("Predictor value")
ax.set_ylabel("Response value")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Robust Sparse**</font>


In this lecture, you learned to:
- Apply sparse linear estimators for feature selection and compact prediction. 
- Use Bayesian linear models to estimate regularized coefficients and uncertainty-related behavior. 
- Fit robust regressors that reduce sensitivity to outliers. 

In the next Lecture (Lecture B), we will go over 'Generalized Bases'